# 01 · Wie lernt eine Maschine?

In diesem Notebook baust du in 20 Minuten ein neuronales Netz, trainierst es und schaust ihm dabei zu, wie es lernt.

**So funktioniert das Notebook**

- Jede graue Box ist eine Zelle. **Shift + Enter** führt sie aus und springt zur nächsten.
- Führe die Zellen **von oben nach unten** aus – jede baut auf der vorherigen auf.
- Zellen mit 🔧 sind zum Drehen gedacht: Werte ändern, nochmal ausführen, schauen, was passiert.
- Wenn etwas komplett hängt: oben rechts *Restart* und wieder von oben starten.

> Das Netz, das du hier baust, funktioniert nach **exakt demselben Prinzip** wie ChatGPT, Copilot & Co. – nur hat es 65 statt 100.000.000.000 Gewichte. Dazu am Ende mehr.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from workshop_utils import (lade_monde, lade_spirale, teile_auf, plot_daten,
                            plot_entscheidung, plot_verlauf, zaehle_parameter,
                            genauigkeit)

torch.manual_seed(0)
print("PyTorch", torch.__version__, "läuft. Los geht's.")

## 1 · Die Daten

Maschinelles Lernen beginnt immer mit Beispielen. Unsere Beispiele sind 400 Punkte mit je zwei Zahlen (x- und y-Koordinate) und einem Etikett: **Klasse 0** oder **Klasse 1**.

In der echten Welt wären die zwei Zahlen vielleicht *Alter eines Tickets* und *Anzahl betroffener Nutzer* – und das Etikett *eskalieren / nicht eskalieren*. Bei Bildern wären es Millionen Pixelwerte, bei Text Tausende Zahlen pro Wort. **Das Prinzip bleibt gleich: Zahlen rein, Entscheidung raus.**

In [ ]:
X, y = lade_monde(n=400, rauschen=0.2)

print("Form von X:", tuple(X.shape), "→ 400 Punkte mit je 2 Zahlen")
print("Form von y:", tuple(y.shape), "→ 400 Etiketten (0 oder 1)")
print("\nDie ersten fünf Punkte:")
for punkt, etikett in zip(X[:5], y[:5]):
    print(f"  x = {punkt[0]:6.2f}   y = {punkt[1]:6.2f}   →  Klasse {int(etikett)}")

plot_daten(X, y);

**Die Aufgabe für das Modell:** Finde eine Grenze, die blau von orange trennt – und zwar so, dass sie auch für *neue* Punkte stimmt, die es noch nie gesehen hat.

Mit einer geraden Linie geht das offensichtlich nicht. Das Modell muss eine krumme Grenze lernen.

## 2 · Das Modell

Ein neuronales Netz ist eine Rechenvorschrift mit **Drehreglern (Gewichten)**. Unser Netz:

- **Eingang:** 2 Zahlen (die Koordinaten)
- **Versteckte Schicht:** 16 Neuronen – jedes rechnet `Eingänge × Gewichte + Bias` und schneidet negative Werte ab (`ReLU`)
- **Ausgang:** 1 Zahl – positiv heißt „Klasse 1", negativ „Klasse 0"

Das Ganze ist in PyTorch drei Zeilen.

In [ ]:
def neues_modell(hidden=16):
    return nn.Sequential(
        nn.Linear(2, hidden),   # 2 Eingänge → hidden Neuronen
        nn.ReLU(),              # negative Werte abschneiden
        nn.Linear(hidden, 1),   # hidden Neuronen → 1 Ausgang
    )

model = neues_modell(hidden=16)
print(model)
print(f"\nDieses Netz hat {zaehle_parameter(model)} Gewichte (lernbare Zahlen).")
print("Zum Vergleich: Ein großes Sprachmodell hat mehrere hundert Milliarden.")

Direkt nach dem Bauen sind alle Gewichte **zufällig**. Das Netz hat noch nichts gelernt – schauen wir uns an, was es *jetzt* sagen würde.

Die Farbe der Fläche zeigt: *Was würde das Modell an dieser Stelle antworten?* Die schwarze Linie ist die Entscheidungsgrenze.

In [ ]:
plot_entscheidung(model, X, y, titel=f"Untrainiert – Genauigkeit {genauigkeit(model, X, y):.0%}");

## 3 · Das Training – die Schleife, um die sich alles dreht

Lernen heißt: **Gewichte so lange anpassen, bis der Fehler klein ist.** Und zwar in einer Schleife mit vier Schritten, die für jedes Modell der Welt gleich ist:

| Schritt | Was passiert | Im Code |
|---|---|---|
| 1 | **Vorhersage** für alle Beispiele | `model(X)` |
| 2 | **Fehler messen** – wie weit liegt die Vorhersage daneben? Diese Zahl heißt *Loss* | `loss_fn(...)` |
| 3 | **Schuld verteilen** – welches Gewicht muss in welche Richtung? (Gradient) | `loss.backward()` |
| 4 | **Einen kleinen Schritt** in diese Richtung gehen. Wie groß der Schritt ist, sagt die **Lernrate** | `optimizer.step()` |

Ein Durchlauf über alle Daten heißt **Epoche**. Lies die Funktion einmal in Ruhe – das ist wirklich alles.

In [ ]:
def trainiere(model, X, y, lernrate=0.1, epochen=300, ausgabe=True):
    optimizer = torch.optim.SGD(model.parameters(), lr=lernrate, momentum=0.9)
    loss_fn = nn.BCEWithLogitsLoss()
    verlauf = []

    for epoche in range(epochen):
        logits = model(X)                 # 1. Vorhersage
        loss = loss_fn(logits, y)         # 2. Fehler messen
        optimizer.zero_grad()             #    (alte Gradienten löschen)
        loss.backward()                   # 3. Schuld verteilen
        optimizer.step()                  # 4. Einen Schritt gehen

        verlauf.append(loss.item())
        if ausgabe and epoche % 50 == 0:
            print(f"Epoche {epoche:4d}   Loss {loss.item():.3f}   Genauigkeit {genauigkeit(model, X, y):.0%}")

    return verlauf

In [ ]:
model = neues_modell(hidden=16)
verlauf = trainiere(model, X, y, lernrate=0.1, epochen=300)

fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 4))
plot_verlauf(verlauf, ax=links)
plot_entscheidung(model, X, y, ax=rechts)
plt.tight_layout()

Links: der Fehler sinkt Epoche für Epoche. Rechts: die gelernte Grenze.

**Das ist maschinelles Lernen.** Niemand hat dem Netz gesagt, wie die Grenze aussehen soll – es hat sie allein aus den Beispielen und dem Fehlersignal gefunden.

## 4 · Zeitraffer – dem Netz beim Lernen zuschauen

Wir trainieren nochmal von vorn und halten die Grenze nach 0, 10, 30, 60, 120 und 300 Epochen fest.

In [ ]:
model = neues_modell(hidden=16)
haltepunkte = [0, 10, 30, 60, 120, 300]

fig, achsen = plt.subplots(2, 3, figsize=(12, 7))
bisher = 0
for ax, ziel in zip(achsen.flat, haltepunkte):
    trainiere(model, X, y, lernrate=0.1, epochen=ziel - bisher, ausgabe=False)
    bisher = ziel
    plot_entscheidung(model, X, y, ax=ax,
                      titel=f"Epoche {ziel} · {genauigkeit(model, X, y):.0%} richtig")
plt.tight_layout()

## 5 · 🔧 Jetzt du: Die drei Drehknöpfe

Unten stehen die drei wichtigsten Stellschrauben. Ändere **einen Wert**, führe die Zelle aus, schau dir Loss-Kurve und Grenze an. Dann den nächsten.

**Experiment A – Lernrate (Schrittweite)**
1. `LERNRATE = 0.001` → Was passiert mit der Loss-Kurve? Wie sieht die Grenze nach 300 Epochen aus?
2. `LERNRATE = 30` → Und jetzt? (Tipp: Schau auf die Zahlen an der Loss-Achse.)

**Experiment B – Größe des Netzes**
1. `HIDDEN = 2` → Kann ein Netz mit nur 2 Neuronen die Monde trennen?
2. `HIDDEN = 256` → Wird es mit 256 besser? Oder nur langsamer?

**Experiment C – Trainingsdauer**
1. `EPOCHEN = 30` → zu früh aufgehört?
2. `EPOCHEN = 2000` → bringt Weitertrainieren noch was?

Schreib dir für jedes Experiment **einen Satz** auf: *Was habe ich gesehen?* Wir sammeln das gleich.

In [ ]:
# 🔧 HIER DREHEN ---------------------------------------------------------
LERNRATE = 0.1      # Schrittweite. Ausprobieren: 0.001  /  30
HIDDEN   = 16       # Neuronen in der Mitte. Ausprobieren: 2  /  256
EPOCHEN  = 300      # Anzahl Durchläufe. Ausprobieren: 30  /  2000
# ------------------------------------------------------------------------

torch.manual_seed(0)
model = neues_modell(hidden=HIDDEN)
verlauf = trainiere(model, X, y, lernrate=LERNRATE, epochen=EPOCHEN, ausgabe=False)

fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 4))
plot_verlauf(verlauf, ax=links, titel=f"Loss · Lernrate {LERNRATE} · {HIDDEN} Neuronen")
plot_entscheidung(model, X, y, ax=rechts,
                  titel=f"Nach {EPOCHEN} Epochen · {genauigkeit(model, X, y):.0%} richtig")
plt.tight_layout()

<details>
<summary><b>Auflösung</b> (erst selbst probieren, dann aufklappen)</summary>

- **Lernrate 0.001:** Der Loss sinkt kaum, die Grenze bleibt fast gerade – die Schritte sind zu klein, 300 Epochen reichen nicht. *Zu vorsichtig.*
- **Lernrate 30:** Der Loss explodiert auf Werte in den Tausendern und bleibt danach schlecht, die Grenze ist unsinnig, Genauigkeit ~50 % – Münzwurf. Die Schritte sind so groß, dass das Modell über das Ziel hinausschießt. *Zu mutig.* (Bei 100 wird der Loss `nan` – das Modell ist rechnerisch kaputt. Bei 10 gibt es einen Ausrutscher, von dem es sich halb erholt.)
- **2 Neuronen:** Zwei Neuronen können nur zwei Knicke – die Monde bleiben halb vermischt. Das Modell ist *zu klein für die Aufgabe* (**Underfitting**).
- **256 Neuronen:** Wird nicht wirklich besser, nur langsamer. Mehr Gewichte helfen erst, wenn die Aufgabe schwieriger wird – und bringen ein neues Problem mit (nächster Abschnitt).
- **30 Epochen:** Grenze noch grob. **2000 Epochen:** Kaum besser als 300 – irgendwann ist gelernt, was zu lernen ist.

</details>

## 6 · Auswendig lernen ist nicht verstehen – Overfitting

Bis jetzt haben wir immer auf denselben Daten trainiert *und* gemessen. Das ist, als würde man die Prüfung mit den Übungsaufgaben schreiben.

Jetzt machen wir es richtig: Wir **verstecken die Hälfte der Daten** (Testdaten) und schauen nach dem Training, wie gut das Modell auf Punkten ist, die es **nie gesehen hat**. Dazu nehmen wir verrauschtere Daten und ein sehr großes Netz.

In [ ]:
X_alle, y_alle = lade_monde(n=300, rauschen=0.35, seed=3)
X_train, y_train, X_test, y_test = teile_auf(X_alle, y_alle, anteil_test=0.5)

fig, achsen = plt.subplots(1, 2, figsize=(11, 4))
for ax, hidden, epochen in zip(achsen, [8, 512], [400, 4000]):
    torch.manual_seed(0)
    model = neues_modell(hidden=hidden)
    trainiere(model, X_train, y_train, lernrate=0.1, epochen=epochen, ausgabe=False)
    plot_entscheidung(model, X_test, y_test, ax=ax,
        titel=f"{hidden} Neuronen · Training {genauigkeit(model, X_train, y_train):.0%} · "
              f"NEUE Daten {genauigkeit(model, X_test, y_test):.0%}")
plt.suptitle("Gezeigt werden jeweils die Testpunkte, die das Modell nie gesehen hat")
plt.tight_layout()

Das große Netz hat auf den Trainingsdaten fast alles richtig – es hat jeden einzelnen Ausreißer **auswendig gelernt** und die Grenze um ihn herumgebogen. Auf neuen Daten ist es deshalb *nicht* besser, oft schlechter. Das heißt **Overfitting**.

Genau deshalb gibt es bei jedem ernsthaften KI-Projekt getrennte Trainings- und Testdaten – und genau deshalb kann ein Sprachmodell Dinge „wissen", die es nur auswendig gelernt hat, ohne sie zu verstehen.

## 7 · 🏆 Challenge (wer noch Zeit hat)

**Die Spirale.** Ein deutlich schwierigerer Datensatz. Schaffst du **mindestens 95 %** auf den Testdaten?

Ideen: mehr Neuronen, mehr Epochen, andere Lernrate – oder eine **zweite versteckte Schicht** (dann hat das Netz mehr Knicke). Beispiel für zwei Schichten:

```python
nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
```

In [ ]:
X_sp, y_sp = lade_spirale()
X_train, y_train, X_test, y_test = teile_auf(X_sp, y_sp, anteil_test=0.3)

# 🔧 Dein Modell -----------------------------------------------------------
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
verlauf = trainiere(model, X_train, y_train, lernrate=0.1, epochen=500, ausgabe=False)
# ------------------------------------------------------------------------

print(f"Gewichte: {zaehle_parameter(model)}")
print(f"Training: {genauigkeit(model, X_train, y_train):.0%}   NEUE Daten: {genauigkeit(model, X_test, y_test):.0%}")
plot_entscheidung(model, X_test, y_test, titel="Spirale – Testdaten");

## 8 · Was hat das mit ChatGPT zu tun?

**Alles.** Ein Sprachmodell ist dieselbe Schleife – Vorhersage, Fehler messen, Schuld verteilen, Schritt gehen – nur in anderer Größenordnung:

| | Unser Netz | Großes Sprachmodell |
|---|---|---|
| Aufgabe | Punkt → blau oder orange? | Text → welches Wort kommt als Nächstes? |
| Gewichte | 65 | einige hundert Milliarden |
| Trainingsdaten | 400 Punkte | Billionen Wörter aus dem Internet |
| Hardware | 2 CPU-Kerne im Codespace | Zehntausende GPUs |
| Trainingsdauer | Sekunden | Monate |
| Die Schleife | `trainiere()` oben | dieselbe |

Alles, was du heute gesehen hast, gilt dort auch: zu große Lernrate → Training explodiert. Zu kleines Modell → Underfitting. Auswendig gelernte Trainingsdaten → das Modell „weiß" Dinge, die es nicht versteht.

➡️ Wenn du sehen willst, wie aus „Punkte trennen" „Text schreiben" wird: **`02_bonus_mini_sprachmodell.ipynb`** – da trainierst du in zwei Minuten ein winziges Sprachmodell, das Deutsch zu schreiben versucht.